In [2]:
!pip install -q anthropic

In [31]:
from anthropic import Anthropic
from google.colab import userdata
from pydantic import BaseModel, Field

anthropic_api_key = userdata.get("ANTHROPIC_API_KEY")
anthropic_client = Anthropic(api_key=anthropic_api_key)

In [6]:
anthropic_models = anthropic_client.models.list()

In [7]:
anthropic_models.data

[ModelInfo(id='claude-sonnet-5', capabilities=ModelCapabilities(batch=CapabilitySupport(supported=True), citations=CapabilitySupport(supported=True), code_execution=CapabilitySupport(supported=True), context_management=ContextManagementCapability(clear_thinking_20251015=CapabilitySupport(supported=True), clear_tool_uses_20250919=CapabilitySupport(supported=True), compact_20260112=CapabilitySupport(supported=True), supported=True), effort=EffortCapability(high=CapabilitySupport(supported=True), low=CapabilitySupport(supported=True), max=CapabilitySupport(supported=True), medium=CapabilitySupport(supported=True), supported=True, xhigh=CapabilitySupport(supported=True)), image_input=CapabilitySupport(supported=True), pdf_input=CapabilitySupport(supported=True), structured_outputs=CapabilitySupport(supported=True), thinking=ThinkingCapability(supported=True, types=ThinkingTypes(adaptive=CapabilitySupport(supported=True), enabled=CapabilitySupport(supported=False)))), created_at=datetime.da

In [11]:
# Chat Completion, Responses -> Open AI
# Messages -> Anthropic

hello_response = anthropic_client.messages.create(
    max_tokens=100,
    model="claude-haiku-4-5-20251001",
    messages = [
        {"role": "user", "content":"Hello!"}
    ]
)

In [12]:
hello_response

Message(id='msg_01D88kaXtLGCfUwtSNdnkzfb', container=None, content=[TextBlock(citations=None, text='Hello! 👋 How can I help you today?', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=9, output_tokens=16, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [19]:
introduction_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    messages =[
        { "role": "user", "content": "Hello! My name is Nikol. What is you name? Please plan my day." }
    ],
    max_tokens = 1024
)

In [20]:
introduction_response

Message(id='msg_01NBYPBRTW8dHgYVUrAo9unG', container=None, content=[TextBlock(citations=None, text="# Hello, Nikol! 👋\n\nI'm Claude, an AI assistant made by Anthropic. Nice to meet you!\n\nI'd be happy to help plan your day, but I'll need some information first:\n\n1. **What time do you wake up and what time do you go to bed?**\n2. **What tasks or activities do you need to accomplish today?** (work, errands, appointments, etc.)\n3. **How much free time do you have?**\n4. **Are there any fixed commitments?** (meetings, classes, appointments with specific times)\n5. **What are your priorities?** (what matters most?)\n6. **How much time do you want for breaks/meals?**\n\nOnce you give me these details, I can create a structured daily schedule for you! 📅", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_t

In [29]:
def simplify(response):
  return {"role": "assistant", "content": response.content}


acquinstance_response = anthropic_client.messages.create(
    model="claude-haiku-4-5-20251001",
    messages =[
        { "role": "user", "content": "Hello! My name is Nikol. What is you name? Please plan my day." },
        simplify(introduction_response),
        { "role": "user", "content": "Hello! My name is Nikol. Do you know my name?"}

    ],
    max_tokens = 1024
)

In [30]:
acquinstance_response

Message(id='msg_0186MrDhui7uewt8mZXcr2hc', container=None, content=[TextBlock(citations=None, text="Yes, I do! You told me your name is **Nikol** in your first message. 😊\n\nHowever, I should mention that I don't retain information between separate conversations. So if you come back and chat with me another day, I won't remember that your name is Nikol unless you tell me again.\n\nBut within *this* conversation, I definitely know your name and can use it anytime!\n\nIs there anything else you'd like help with today?", type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=228, output_tokens=104, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

## Structured Output

In [32]:
class CalendarEvent(BaseModel):
  name: str
  short_description: str
  duration_h: float = Field(description="Duration of the event in hours.")

In [40]:
structured_response=anthropic_client.messages.parse(
    model="claude-haiku-4-5-20251001",
    messages=[
        {"role":"user","content":"Generate a random calendar event for thursday morning."}
    ],
    system = "You are helpful agent.",
    max_tokens = 1024,
    output_format= CalendarEvent
)

In [41]:
structured_response

ParsedMessage[TypeVar](id='msg_01Kmjj47NN28DYRPUhqRUWY4', container=None, content=[ParsedTextBlock[TypeVar](citations=None, text='{"name": "Team Standup Meeting", "short_description": "Weekly sync with the development team to discuss project progress and blockers", "duration_h": 0.5}', type='text', parsed_output=CalendarEvent(name='Team Standup Meeting', short_description='Weekly sync with the development team to discuss project progress and blockers', duration_h=0.5))], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=264, output_tokens=43, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [37]:
structured_response.parsed_output

CalendarEvent(name='Team Standup Meeting', short_description='Weekly sync with the development team to discuss progress, blockers, and upcoming tasks for the sprint.', duration_h=0.5)

## Streaming

In [39]:
with anthropic_client.messages.stream(
    model="claude-haiku-4-5-20251001",
    messages = [
        {"role":"user","content":"Describe all popular types of pizza with their ingredients and reate them by best flavour."}
    ],
    max_tokens = 4096
) as basic_stream:
    for event in basic_stream.text_stream:
        print(event, end='', flush=True)

# Popular Pizza Types Ranked by Flavor

## 1. **Neapolitan (Classic)**
**Ingredients:** Tomato sauce, fresh mozzarella, basil, olive oil, salt
- The original, perfected over centuries. Simple but sublime—the quality of each ingredient shines.

## 2. **Margherita (Variation)**
**Ingredients:** Tomato sauce, mozzarella, fresh basil, parmesan, olive oil
- Nearly identical to Neapolitan with slightly more complexity from parmesan.

## 3. **Prosciutto & Arugula (Prosciutto e Rucola)**
**Ingredients:** Tomato base, mozzarella, prosciutto, fresh arugula, parmesan, olive oil
- The peppery arugula and salty prosciutto create perfect balance.

## 4. **BBQ Chicken**
**Ingredients:** BBQ sauce, chicken, red onions, cilantro, mozzarella, cheddar
- Bold, smoky, and slightly sweet—highly satisfying comfort food.

## 5. **Quattro Formaggi (Four Cheese)**
**Ingredients:** Mozzarella, gorgonzola, parmesan, ricotta
- Rich and decadent; cheese lovers' paradise.

## 6. **Capricciosa**
**Ingredients:** Toma